# Chapter 14 — Collaboration

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 14.

---


### Item 116 — Know Where to Find Community-Built Modules

The Python Package Index (PyPI, at pypi.org) hosts the vast majority of third-party Python packages, installed via `pip install package_name`. Before writing something non-trivial from scratch, it's worth checking whether a well-maintained package already solves it — `pip show package_name` reveals what's already installed and where it lives, without needing to open a browser.


In [1]:
import subprocess

# pip show reveals metadata about an already-installed package -- version, location, dependencies
result = subprocess.run(['pip', 'show', 'requests'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'requests not installed in this environment')

# pip list shows everything currently installed -- a quick inventory of what's already available
result2 = subprocess.run(['pip', 'list'], capture_output=True, text=True)
print('\n'.join(result2.stdout.splitlines()[:8]), '\n... (truncated)')


Name: requests
Version: 2.34.2
Summary: Python HTTP for Humans.
Home-page: 
Author: 
Author-email: Kenneth Reitz <me@kennethreitz.org>
License: Apache-2.0
Location: /usr/local/lib/python3.10/dist-packages
Requires: certifi, charset_normalizer, idna, urllib3
Required-by: markitdown



Package                   Version
------------------------- ----------------
altgraph                  0.17.5
argcomplete               1.8.1
asttokens                 3.0.1
attrs                     26.1.0
Automat                   20.2.0
Babel                     2.8.0 
... (truncated)


### Item 117 — Use Virtual Environments for Isolated and Reproducible Dependencies

`python -m venv myenv` creates an isolated Python environment with its own separate `site-packages`, so a project's exact dependency versions don't clash with another project's (or the system's) — and `pip freeze > requirements.txt` captures the exact installed versions so a teammate (or a deployment server) can reproduce the identical environment with `pip install -r requirements.txt`.


In [2]:
import subprocess, sys, pathlib, shutil

venv_dir = pathlib.Path('collab_venv')
subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)

venv_pip = venv_dir / 'bin' / 'pip'
subprocess.run([str(venv_pip), 'install', '--quiet', 'six'], check=True)   # a tiny, fast-installing package

# Freeze this isolated environment's EXACT versions -- reproducible on any other machine
freeze = subprocess.run([str(venv_pip), 'freeze'], capture_output=True, text=True)
print('requirements.txt contents for this isolated environment:')
print(freeze.stdout)

shutil.rmtree(venv_dir)


requirements.txt contents for this isolated environment:
six==1.17.0



### Item 118 — Write Docstrings for Every Function, Class, and Module

A docstring (the first statement in a function/class/module, as a string literal) is retrievable at runtime via `.__doc__` or `help()` — unlike a `#` comment, tools (IDEs, `help()`, Sphinx documentation generators) surface it automatically. The convention: describe what it does, its arguments, its return value, and any exceptions it raises — enough that a caller never needs to read the implementation to use it correctly.


In [3]:
def calculate_discount(price, percent):
    """Calculate the discounted price.

    Args:
        price: The original price, as a float.
        percent: The discount percentage (e.g. 20 for 20% off).

    Returns:
        The discounted price as a float.

    Raises:
        ValueError: If percent is not between 0 and 100.
    """
    if not 0 <= percent <= 100:
        raise ValueError('percent must be between 0 and 100')
    return price * (1 - percent / 100)

print(calculate_discount(100, 20))
print(calculate_discount.__doc__)     # retrievable at runtime, unlike a plain comment
help(calculate_discount)               # help() formats it nicely, using the SAME docstring


80.0
Calculate the discounted price.

    Args:
        price: The original price, as a float.
        percent: The discount percentage (e.g. 20 for 20% off).

    Returns:
        The discounted price as a float.

    Raises:
        ValueError: If percent is not between 0 and 100.
    
Help on function calculate_discount in module __main__:

calculate_discount(price, percent)
    Calculate the discounted price.
    
    Args:
        price: The original price, as a float.
        percent: The discount percentage (e.g. 20 for 20% off).
    
    Returns:
        The discounted price as a float.
    
    Raises:
        ValueError: If percent is not between 0 and 100.



### Item 119 — Use Packages to Organize Modules and Provide Stable APIs

A package's `__init__.py` can act as a curated "public API" for everything inside it: explicitly importing and re-exporting the pieces meant for external use (and optionally defining `__all__`) lets internal module structure be reorganized freely later without breaking anyone who only ever imported from the top-level package name.


In [4]:
from pathlib import Path
import sys, shutil

pkg = Path('demo_toolkit')
pkg.mkdir(exist_ok=True)

(pkg / '_internal_math.py').write_text('def add(a, b):\n    return a + b\n')
(pkg / '_internal_strings.py').write_text('def shout(s):\n    return s.upper() + "!"\n')

# __init__.py curates a STABLE public API, hiding which internal module each piece actually lives in
(pkg / '__init__.py').write_text(
    'from demo_toolkit._internal_math import add\n'
    'from demo_toolkit._internal_strings import shout\n'
    '__all__ = ["add", "shout"]\n'
)

sys.path.insert(0, '.')
import demo_toolkit
print(demo_toolkit.add(2, 3))       # caller never needs to know it actually lives in _internal_math
print(demo_toolkit.shout('hi'))
print(demo_toolkit.__all__)

del sys.modules['demo_toolkit']
for mod in list(sys.modules):
    if mod.startswith('demo_toolkit'):
        del sys.modules[mod]
shutil.rmtree(pkg)


5
HI!
['add', 'shout']


### Item 120 — Consider Module-Scoped Code to Configure Deployment Environments

Code that runs at **import time** (module-level, not inside a function) can inspect the deployment environment and configure module-level state accordingly — e.g., picking a real vs. a fake/test client based on an environment variable — so the rest of the codebase just imports the already-correctly-configured object, without every call site needing its own environment-detection logic.


In [5]:
import os

# Module-scope code (runs once, at import time) picks the right implementation for this environment
if os.environ.get('APP_ENV') == 'production':
    DATABASE_URL = 'postgres://prod-server/db'
else:
    DATABASE_URL = 'sqlite:///local_dev.db'   # safe default for this demo

print('Configured for this environment at import time:', DATABASE_URL)

# The rest of the code just uses DATABASE_URL directly -- no repeated environment checks needed elsewhere
def connect():
    return f'connecting to {DATABASE_URL}'

print(connect())


Configured for this environment at import time: sqlite:///local_dev.db
connecting to sqlite:///local_dev.db


### Item 121 — Define a Root `Exception` to Insulate Callers from APIs

A library/module that only ever raises its own custom base exception (with more specific exceptions subclassing it) lets callers write one `except MyLibraryError:` to catch anything the library can raise, without needing to know or list every specific exception type — and it cleanly separates "this came from my library" from unrelated exceptions elsewhere in their code.


In [6]:
class MyLibraryError(Exception):
    """Root exception for this library -- callers can catch just this to handle ANY library failure."""
    pass

class InvalidInputError(MyLibraryError):
    pass

class ConnectionFailedError(MyLibraryError):
    pass

def process(value):
    if value < 0:
        raise InvalidInputError(f'{value} is negative')
    if value > 1000:
        raise ConnectionFailedError('simulated failure for large values')
    return value * 2

# Callers can catch the ROOT exception, without needing to enumerate every specific subclass
for test_value in [5, -1, 2000]:
    try:
        print(process(test_value))
    except MyLibraryError as e:
        print(f'Handled (any library failure): {type(e).__name__}: {e}')


10
Handled (any library failure): InvalidInputError: -1 is negative
Handled (any library failure): ConnectionFailedError: simulated failure for large values


### Item 122 — Know How to Break Circular Dependencies

Two modules that each need to `import` the other at the top of the file create a circular import, which raises `ImportError` the moment either is imported first. Three standard fixes: reorder so the dependency is one-directional, move the shared code both need into a third, independent module, or move one of the imports inside a function body (deferring it until it's actually called, by which point both modules have finished loading).


In [7]:
from pathlib import Path
import sys, shutil

pkg = Path('circular_demo')
pkg.mkdir(exist_ok=True)
(pkg / '__init__.py').write_text('')

# The FIX shown here: defer the import inside the function, instead of at module top-level
(pkg / 'module_a.py').write_text(
    'def use_b():\n'
    '    from circular_demo import module_b   # deferred import -- avoids the circular failure at load time\n'
    '    return module_b.greeting() + " from A"\n'
)
(pkg / 'module_b.py').write_text(
    'def greeting():\n'
    '    return "hello"\n'
    'def use_a():\n'
    '    from circular_demo import module_a   # deferred here too, since both reference each other\n'
    '    return module_a.use_b() + " (via B)"\n'
)

sys.path.insert(0, '.')
from circular_demo import module_a, module_b
print(module_a.use_b())
print(module_b.use_a())

for mod in list(sys.modules):
    if mod.startswith('circular_demo'):
        del sys.modules[mod]
shutil.rmtree(pkg)


hello from A
hello from A (via B)


### Item 123 — Consider `warnings` to Refactor and Migrate Usage

When deprecating an old function/parameter in favor of a new one, the standard library's `warnings.warn(msg, DeprecationWarning)` tells callers their code still works *today* but should migrate — visible to developers running with warnings enabled (and to static analysis tools), without breaking anyone immediately the way outright removal would.


In [8]:
import warnings

def old_function(x):
    warnings.warn(
        'old_function is deprecated, use new_function instead',
        DeprecationWarning,
        stacklevel=2
    )
    return new_function(x)

def new_function(x):
    return x * 2

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')   # ensure the warning is captured for this demo
    result = old_function(5)
    print('result:', result)
    print('warning captured:', str(caught[0].message))
    print('warning category:', caught[0].category.__name__)


result: 10
warning captured: old_function is deprecated, use new_function instead
warning category: DeprecationWarning


### Item 124 — Consider Static Analysis via `typing` to Obviate Bugs

Type hints (`def add(a: int, b: int) -> int:`) don't affect runtime behavior at all — Python never enforces them on its own — but tools like `mypy` analyze them **before** the code ever runs, catching a real class of bugs (passing the wrong type, calling a method that doesn't exist on that type) at development time instead of in production.


In [9]:
from typing import Optional

def find_user(user_id: int) -> Optional[str]:
    """Returns the user's name, or None if not found."""
    users = {1: 'Ada', 2: 'Alan'}
    return users.get(user_id)

# Type hints don't stop this from RUNNING at all -- Python itself never checks them
name = find_user(1)
print(name)

wrong_call = find_user('not an int')   # runs fine at runtime -- but a type checker like mypy WOULD flag this
print(wrong_call)

print(find_user.__annotations__)   # the hints ARE introspectable at runtime, just not enforced
print('Static analysis tools (mypy, pyright) read these annotations to catch mismatches before runtime.')


Ada
None
{'user_id': <class 'int'>, 'return': typing.Optional[str]}
Static analysis tools (mypy, pyright) read these annotations to catch mismatches before runtime.


### Item 125 — Prefer Open Source Projects for Bundling Python Programs over `zipimport` and `zipapp`

Python can technically import modules straight out of a `.zip` file (`zipimport`) and `zipapp` can bundle a whole program plus its dependencies into one runnable `.pyz` archive — but for real distribution, the book recommends established, actively-maintained community tools (like PyInstaller, covered in the ATBS track, or Shiv/PEX) instead: they handle platform-specific quirks, native dependencies, and edge cases that a hand-rolled zipapp bundle typically doesn't.


In [10]:
import zipapp
import subprocess
import sys
from pathlib import Path
import shutil

app_dir = Path('zipapp_demo')
app_dir.mkdir(exist_ok=True)
(app_dir / '__main__.py').write_text("print('hello from a .pyz zipapp bundle')\n")

zipapp.create_archive(str(app_dir), target='demo.pyz', interpreter=sys.executable)

result = subprocess.run([sys.executable, 'demo.pyz'], capture_output=True, text=True)
print('Running the bundled .pyz:', result.stdout.strip())

print()
print('zipapp works for simple, pure-Python cases like this one -- but for real distribution')
print('with native dependencies, PyInstaller (see the ATBS track, chapter 12) is the more robust,')
print('actively-maintained choice the book recommends over hand-rolling zipimport/zipapp bundles.')

shutil.rmtree(app_dir)
Path('demo.pyz').unlink()


Running the bundled .pyz: hello from a .pyz zipapp bundle

zipapp works for simple, pure-Python cases like this one -- but for real distribution
with native dependencies, PyInstaller (see the ATBS track, chapter 12) is the more robust,
actively-maintained choice the book recommends over hand-rolling zipimport/zipapp bundles.
